# Court keypoint model: ResNet50 -> 14 (x, y) points

A corrected version of the upstream notebook, which does not execute as
published. Four bugs, and what each one cost:

| bug | effect |
| --- | --- |
| `items['kps']` (should be `item`) | `NameError` on the first batch |
| `model.stat_dict()` (should be `state_dict()`) | `AttributeError` on the **last** line, after all the training time was spent. Nothing was saved. |
| `devic = torch.device(...)` | the typo'd name is never used, so `model.to(device)` raises |
| `val_loader` built, never iterated | no validation loss, no metric, no early stopping |

The last is the substantive one. A regression model trained for a fixed
epoch count with no validation signal is not a trained model, it is a model
that stopped. There is no way to know whether it overfitted 20 epochs ago.

For real runs prefer the script, which does the same thing with
checkpointing and a proper report:

```bash
python training/train_keypoints.py --data-dir data --epochs 60
```


## Dataset

The upstream notebook fetched this with a `wget` carrying the author's live
Google session cookie, which has been redacted from this repo. Supply your
own copy laid out as:

```
data/
  images/          <id>.png
  data_train.json  [{"id": ..., "kps": [[x, y], ... 14 pairs]}, ...]
  data_val.json
```


In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

NUM_KEYPOINTS = 14
INPUT_SIZE = 224
DATA_DIR = Path('data')

# The typo'd name in the upstream notebook was `devic`, so every later
# `.to(device)` raised NameError.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

## Dataset class

Note the direction of the keypoint scaling: labels are scaled **into**
224x224 space here, and predictions are scaled back **out** at inference.
Getting those two out of step is the classic way to end up with a model that
looks trained and predicts nonsense.


In [ ]:
class KeypointsDataset(Dataset):
    def __init__(self, image_dir, annotation_file):
        self.image_dir = Path(image_dir)
        with open(annotation_file, encoding='utf-8') as f:
            self.items = json.load(f)

        self.transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]          # upstream wrote `items[...]` -> NameError
        img = cv2.imread(f"{self.image_dir}/{item['id']}.png")
        if img is None:
            raise FileNotFoundError(item['id'])
        h, w = img.shape[:2]

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(img)

        kps = np.array(item['kps'], dtype=np.float32).flatten()
        kps[0::2] *= INPUT_SIZE / w     # x
        kps[1::2] *= INPUT_SIZE / h     # y
        return img, torch.from_numpy(kps)

In [ ]:
train_dataset = KeypointsDataset(DATA_DIR / 'images', DATA_DIR / 'data_train.json')
val_dataset = KeypointsDataset(DATA_DIR / 'images', DATA_DIR / 'data_val.json')

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
# shuffle=False for validation: the metric must not depend on ordering, and a
# fixed order makes runs comparable. Upstream shuffled here too.
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(len(train_dataset), 'train |', len(val_dataset), 'val')

## Model

ResNet50 with its 1000-class classifier replaced by a 28-value regression
head - 14 keypoints x (x, y). ImageNet features transfer well: court lines
are edges and corners, which the early layers already detect.


In [ ]:
# `weights=` rather than the deprecated `pretrained=True`.
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = torch.nn.Linear(model.fc.in_features, NUM_KEYPOINTS * 2)
model = model.to(device)

## Metric

PCK - Percentage of Correct Keypoints - is the share of predictions landing
within a threshold of the truth, the threshold given as a fraction of image
size so it is resolution independent. PCK@0.05 at 224 px means within ~11 px.

Raw MSE cannot be compared across resolutions and gives no intuition about
whether a court would actually be usable. Upstream reported neither.


In [ ]:
def pck(pred, target, threshold=0.05):
    pred = pred.reshape(-1, NUM_KEYPOINTS, 2)
    true = target.reshape(-1, NUM_KEYPOINTS, 2)
    d = torch.linalg.norm(pred - true, dim=2)
    return float((d < threshold * INPUT_SIZE).float().mean())


@torch.no_grad()
def evaluate(model, loader, criterion):
    """The validation pass upstream built a loader for and never ran."""
    model.eval()
    total, preds, trues = 0.0, [], []
    for imgs, kps in loader:
        imgs, kps = imgs.to(device), kps.to(device)
        out = model(imgs)
        total += float(criterion(out, kps)) * imgs.size(0)
        preds.append(out.cpu())
        trues.append(kps.cpu())

    p, t = torch.cat(preds), torch.cat(trues)
    px = torch.linalg.norm(p.reshape(-1, NUM_KEYPOINTS, 2)
                           - t.reshape(-1, NUM_KEYPOINTS, 2), dim=2).mean()
    return {'loss': total / len(loader.dataset),
            'pck_005': pck(p, t, 0.05),
            'pck_010': pck(p, t, 0.10),
            'px_error': float(px)}

## Training

Validating every epoch and keeping the best weights by PCK, rather than
running a fixed 20 epochs and saving whatever the last one happened to be.


In [ ]:
EPOCHS = 60
criterion = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_pck, history = -1.0, []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for imgs, kps in train_loader:
        imgs, kps = imgs.to(device), kps.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), kps)
        loss.backward()
        optimizer.step()
        running += float(loss) * imgs.size(0)

    scheduler.step()
    train_loss = running / len(train_dataset)
    m = evaluate(model, val_loader, criterion)
    history.append({'epoch': epoch, 'train_loss': train_loss, **m})

    flag = ''
    if m['pck_005'] > best_pck:
        best_pck = m['pck_005']
        # state_dict(), not stat_dict(); and saved on every improvement, so a
        # crash at epoch 55 does not discard the entire run.
        torch.save(model.state_dict(), 'keypoints_model.pth')
        flag = '  <- best, saved'

    print(f"epoch {epoch:3d}  train {train_loss:8.2f}  val {m['loss']:8.2f}  "
          f"PCK@0.05 {m['pck_005']:.3f}  px {m['px_error']:5.2f}{flag}")

print('best PCK@0.05:', round(best_pck, 4))

## Training curves


In [ ]:
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4))
a.plot(epochs, [h['train_loss'] for h in history], label='train')
a.plot(epochs, [h['loss'] for h in history], label='val')
a.set(xlabel='epoch', ylabel='MSE', title='Loss'); a.legend(); a.set_yscale('log')
b.plot(epochs, [h['pck_005'] for h in history], label='PCK@0.05')
b.plot(epochs, [h['pck_010'] for h in history], label='PCK@0.10')
b.set(xlabel='epoch', ylabel='fraction correct', title='Accuracy'); b.legend()
plt.tight_layout(); plt.show()

# A val curve that flattens or rises while train keeps falling is the
# overfitting the upstream notebook had no way to see.